# House Prices v4 — 하이브리드: 구조 모델 + Lasso 잔차 보정 (목표 LB 0.114)

지금까지 LB: v2 stack 0.11535 / v3.1 blend 0.11539 / v3.1 stack 0.11545

v4에서 바꾼 것은 하나예요.
- **하이브리드 모델**: 구조 모델(지하 마감 포함)이 먼저 가격을 예측하고, Lasso는 집값 전체가 아니라 **구조 모델이 틀린 부분(잔차)만** 학습해서 보정해요.
  - 사람이 정한 원가법 구조가 큰 틀을 잡고, 302 → 252개 피처는 그 수식이 놓친 부분만 메우는 역할
  - 단독 CV 0.1054: Lasso 0.1092, 구조 모델 0.1103보다 크게 좋음. 바깥 분할 3가지에서도 Lasso보다 매번 0.0013~0.0023 좋음
- 스택: XGBoost + SVR + 하이브리드 (Lasso와 구조 모델 단독은 하이브리드가 들어오면 가중치가 0 근처라 제외)
  - 스택 CV 0.1043 → **0.1034**, 단순 평균 0.1044 → 0.1036

같이 시험했지만 효과 없던 것: CatBoost(범주형 그대로, 가중치 0), v2 stack과 v3.1 blend 제출 평균(0.1045)

주의: 하이브리드는 test 예측이 Lasso와 10% 넘게 다른 집이 66채이고, 상위권이 IDOTRR 동네에 몰려 있어요(구조 모델 단독은 111채). 구조 모델의 IDOTRR 약점(땅값·건축비 맞바뀜)을 일부 물려받은 셈이라, CV 개선이 LB로 넘어오는지는 제출로 확인해야 해요.

7-0a, 7-0b(구조 모델 항목 실험)는 결과를 v3에서 기록했으므로 건너뛰도록 해뒀어요. 다시 보려면 첫 줄의 `%%script echo skipped`를 지우면 돼요.

In [1]:
#0 라이브러리 & 데이터
import os, glob, time, warnings
import numpy as np, pandas as pd
warnings.filterwarnings('ignore')

# Kaggle 경로 자동 탐색 (로컬이면 ../data)
cands = glob.glob('/kaggle/input/**/train.csv', recursive=True)
DATA = os.path.dirname(cands[0]) if cands else '../data'
train_raw = pd.read_csv(f'{DATA}/train.csv')
test_raw = pd.read_csv(f'{DATA}/test.csv')
print(DATA, train_raw.shape, test_raw.shape)

/kaggle/input/competitions/house-prices-advanced-regression-techniques (1460, 81) (1459, 80)


In [2]:
#1 전처리 함수 (옵션으로 v1 ↔ v2 비교 가능)
from scipy.stats import skew, boxcox_normmax
from scipy.special import boxcox1p

QUAL = {'None': 0, 'Po': 1, 'Fa': 2, 'TA': 3, 'Gd': 4, 'Ex': 5}

def build(train, test, boxcox=True, more_ord=True, interact=True, drop_rare=True):
    # 1. 이상치 제거 + 타깃 로그 변환
    train = train[~train['Id'].isin([524, 1299])].reset_index(drop=True)
    y = np.log1p(train['SalePrice'])
    n = len(train)
    a = pd.concat([train.drop(columns=['Id', 'SalePrice']),
                   test.drop(columns=['Id'])]).reset_index(drop=True)

    # 2. 결측치 (v1과 동일)
    for c in ['PoolQC', 'MiscFeature', 'Alley', 'Fence', 'FireplaceQu',
              'GarageType', 'GarageFinish', 'GarageQual', 'GarageCond',
              'BsmtQual', 'BsmtCond', 'BsmtExposure', 'BsmtFinType1', 'BsmtFinType2', 'MasVnrType']:
        a[c] = a[c].fillna('None')
    for c in ['GarageArea', 'GarageCars', 'BsmtFinSF1', 'BsmtFinSF2', 'BsmtUnfSF',
              'TotalBsmtSF', 'BsmtFullBath', 'BsmtHalfBath', 'MasVnrArea']:
        a[c] = a[c].fillna(0)
    a['GarageYrBlt'] = a['GarageYrBlt'].fillna(a['YearBuilt'])
    a.loc[a['GarageYrBlt'] > 2010, 'GarageYrBlt'] = a['YearBuilt']   # test의 2207년 오타
    a['LotFrontage'] = a.groupby('Neighborhood')['LotFrontage'].transform(lambda s: s.fillna(s.median()))
    a['Functional'] = a['Functional'].fillna('Typ')
    for c in ['MSZoning', 'Electrical', 'KitchenQual', 'Exterior1st', 'Exterior2nd', 'SaleType']:
        a[c] = a[c].fillna(a[:n][c].mode()[0])
    a = a.drop(columns=['Utilities'])
    for c in ['MSSubClass', 'MoSold', 'YrSold']:
        a[c] = a[c].astype(str)

    # 3. 파생 변수 (v1과 동일)
    ys = a['YrSold'].astype(int)
    a['TotalSF'] = a['TotalBsmtSF'] + a['1stFlrSF'] + a['2ndFlrSF']
    a['TotalBath'] = a['FullBath'] + 0.5 * a['HalfBath'] + a['BsmtFullBath'] + 0.5 * a['BsmtHalfBath']
    a['HouseAge'] = (ys - a['YearBuilt']).clip(lower=0)
    a['RemodAge'] = (ys - a['YearRemodAdd']).clip(lower=0)
    a['TotalPorchSF'] = (a['OpenPorchSF'] + a['EnclosedPorch'] + a['3SsnPorch']
                         + a['ScreenPorch'] + a['WoodDeckSF'])
    for c in ['PoolArea', '2ndFlrSF', 'GarageArea', 'TotalBsmtSF', 'MasVnrArea', 'Fireplaces',
              'MiscVal', '3SsnPorch', 'LowQualFinSF', 'BsmtFinSF2', 'ScreenPorch', 'EnclosedPorch',
              'WoodDeckSF', 'OpenPorchSF']:
        a['Has' + c] = (a[c] > 0).astype(int)

    # 4. 순서형 인코딩
    for c in ['ExterQual', 'KitchenQual', 'BsmtQual', 'FireplaceQu', 'HeatingQC']:
        a[c] = a[c].map(QUAL)
    a['GarageFinish'] = a['GarageFinish'].map({'None': 0, 'Unf': 1, 'RFn': 2, 'Fin': 3})
    a['BsmtExposure'] = a['BsmtExposure'].map({'None': 0, 'No': 1, 'Mn': 2, 'Av': 3, 'Gd': 4})
    if more_ord:   # [v2] 추가
        for c in ['GarageQual', 'GarageCond', 'BsmtCond', 'PoolQC']:
            a[c] = a[c].map(QUAL)
        a['Functional'] = a['Functional'].map({'Sal': 0, 'Sev': 1, 'Maj2': 2, 'Maj1': 3,
                                               'Mod': 4, 'Min2': 5, 'Min1': 6, 'Typ': 7})
        a['CentralAir'] = (a['CentralAir'] == 'Y').astype(int)
        a['PavedDrive'] = a['PavedDrive'].map({'N': 0, 'P': 1, 'Y': 2})

    # 5. 대부분 0인 컬럼은 유무만 남김 + 희귀 범주 묶기 (v1과 동일)
    a = a.drop(columns=['MiscVal', '3SsnPorch', 'LowQualFinSF', 'BsmtFinSF2',
                        'ScreenPorch', 'EnclosedPorch', 'PoolArea'])
    for c in ['Condition2', 'RoofMatl', 'Heating']:
        cnt = a[:n][c].value_counts()
        a[c] = a[c].where(a[c].isin(cnt[cnt >= 10].index), 'Other')

    # 6. 치우친 연속형 변환
    if boxcox:   # [v2] 왜도 0.5 이상인 연속형 전부 Box-Cox, λ는 train으로만 추정
        num = [c for c in a.select_dtypes(include='number').columns
               if a[c].nunique() > 15 and not c.startswith('Has')
               and c not in ('YearBuilt', 'YearRemodAdd', 'GarageYrBlt')]
        sk = a[:n][num].apply(lambda s: skew(s))
        for c in sk[sk.abs() > 0.5].index:
            if (a[c] < 0).any():
                continue
            try:
                lam = boxcox_normmax(a[:n][c] + 1, method='mle')
            except Exception:
                lam = 0.0   # 추정 실패하면 로그 변환과 같음
            a[c] = boxcox1p(a[c], lam)
    else:        # v1 방식
        for c in ['LotArea', 'LotFrontage', 'BsmtFinSF1', 'BsmtUnfSF', 'TotalBsmtSF',
                  '1stFlrSF', 'GrLivArea', 'TotalSF', 'MasVnrArea', 'TotalPorchSF']:
            a[c] = np.log1p(a[c])

    if interact:   # [v2] 기본 탑재
        a['Qual_SF'] = a['OverallQual'] * a['TotalSF']
        a['Qual_Liv'] = a['OverallQual'] * a['GrLivArea']

    # 7. 원-핫 + 희귀 더미 제거
    a = pd.get_dummies(a).astype(float)
    if drop_rare:   # [v2] train에서 5채 미만만 다른 값 → 과적합 재료
        diff = (a[:n] != a[:n].mode().iloc[0]).sum()
        a = a.drop(columns=diff[diff < 5].index)

    X, X_test = a[:n].reset_index(drop=True), a[n:].reset_index(drop=True)
    return X, X_test, y, train

X, X_test, y, train = build(train_raw, test_raw)
print(X.shape, X_test.shape, '결측:', X.isna().sum().sum() + X_test.isna().sum().sum())

(1458, 252) (1459, 252) 결측: 0


In [3]:
#2 검증 도구
from sklearn.model_selection import KFold, RepeatedKFold, cross_val_score, cross_val_predict
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import RobustScaler

kf = KFold(n_splits=5, shuffle=True, random_state=42)                    # OOF용
rkf = RepeatedKFold(n_splits=5, n_repeats=3, random_state=42)           # 피처 비교용 (15번)
rmse = lambda p: np.sqrt(((y.values - p) ** 2).mean())

def cv_scores(model, X_):
    return -cross_val_score(model, X_, y, cv=rkf, scoring='neg_root_mean_squared_error')

## 3. v1 → v2 변경 효과 확인
변경을 하나씩 켜서 Lasso와 SVR로 15번 비교해요. 각 변경 단독 효과는 0.0004~0.001로 작지만, 합치면 Lasso는 15번 중 15번, SVR은 14번 좋아져요. 발표 때 "왜 바꿨는지" 근거로 쓰기 좋은 표예요.

In [4]:
#3 변경 효과 (반복 교차검증 15번, 짝지어 비교)
from sklearn.linear_model import Lasso
from sklearn.svm import SVR

lasso = make_pipeline(RobustScaler(), Lasso(alpha=0.0005, max_iter=50000))
svr = make_pipeline(RobustScaler(), SVR(C=15, epsilon=0.008, gamma=0.0005))

v1 = dict(boxcox=False, more_ord=False, interact=False, drop_rare=False)
X_v1 = build(train_raw, test_raw, **v1)[0]
base = {n: cv_scores(m, X_v1) for n, m in [('Lasso', lasso), ('SVR', svr)]}

rows = []
for name, change in [('Box-Cox', {'boxcox': True}), ('순서형 확대', {'more_ord': True}),
                     ('상호작용', {'interact': True}), ('희귀 더미 제거', {'drop_rare': True}),
                     ('v2 전체', {k: True for k in v1})]:
    Xc = build(train_raw, test_raw, **{**v1, **change})[0]
    row = {'변경': name, '컬럼 수': Xc.shape[1]}
    for n, m in [('Lasso', lasso), ('SVR', svr)]:
        s = cv_scores(m, Xc)
        row[n] = round(s.mean(), 4)
        row[n + ' 개선 횟수'] = f'{(s < base[n]).sum()}/15'
    rows.append(row)
print('v1 기준: Lasso', base['Lasso'].mean().round(4), '| SVR', base['SVR'].mean().round(4))
pd.DataFrame(rows)

v1 기준: Lasso 0.1117 | SVR 0.111


,변경,컬럼 수,Lasso,Lasso 개선 횟수,SVR,SVR 개선 횟수
0,Box-Cox,302,0.1111,9/15,0.1101,10/15
1,순서형 확대,276,0.1108,12/15,0.1103,12/15
2,상호작용,304,0.1114,15/15,0.1108,10/15
3,희귀 더미 제거,268,0.1117,6/15,0.1110,6/15
4,v2 전체,252,0.1093,15/15,0.1088,14/15


In [5]:
#4 구조 모델 v4 (v3 수식 + 켜고 끌 수 있는 항목들)
from scipy.optimize import least_squares

Q = {'None': 0, 'Po': 1, 'Fa': 2, 'TA': 3, 'Gd': 4, 'Ex': 5}
SALE = ['Abnorml', 'Family', 'Alloca', 'Partial', 'AdjLand']
ZONE = ['C (all)', 'RM', 'RH', 'FV']
BLDG = ['2fmCon', 'Duplex', 'Twnhs', 'TwnhsE']
YRS = ['2007', '2008', '2009', '2010']
FOUND = ['CBlock', 'BrkTil', 'Slab', 'Stone', 'Wood']   # 기준: PConc

def prep_struct(d):
    d = d.copy()
    for c in ['TotalBsmtSF', 'GarageArea', 'BsmtFullBath', 'BsmtHalfBath',
              'BsmtFinSF1', 'BsmtFinSF2', 'GarageCars']:
        d[c] = d[c].fillna(0)
    ys = d['YrSold'].astype(int)
    d['age'] = (ys - d['YearBuilt']).clip(lower=0)
    d['remod_age'] = (ys - d['YearRemodAdd']).clip(lower=0)
    d['MSZoning'] = d['MSZoning'].fillna('RL')
    d['KQ'] = d['KitchenQual'].map(Q).fillna(3)
    d['EQ'] = d['ExterQual'].map(Q).fillna(3)
    d['BQ'] = d['BsmtQual'].fillna('None').map(Q)
    d['HQ'] = d['HeatingQC'].map(Q).fillna(3)
    d['FUNC'] = d['Functional'].fillna('Typ').map({'Sal': 0, 'Sev': 1, 'Maj2': 2, 'Maj1': 3,
                                                   'Mod': 4, 'Min2': 5, 'Min1': 6, 'Typ': 7})
    d['bath'] = d['FullBath'] + 0.5 * d['HalfBath'] + d['BsmtFullBath'] + 0.5 * d['BsmtHalfBath']
    d['porch'] = d[['WoodDeckSF', 'OpenPorchSF', 'EnclosedPorch', 'ScreenPorch', '3SsnPorch']].sum(axis=1)
    d['air'] = (d['CentralAir'] == 'Y').astype(float)
    d['bs_fin'] = d['BsmtFinSF1'] + d['BsmtFinSF2']
    d['YrSold_s'] = ys.astype(str)
    return d

class StructModel:
    """가격 = [땅값 + 건물값] x 전체 배수 + 부대시설, 그 위에 거래 조건·용도지역 % 보정.
    opts로 항목을 켜고 끔. opts=()이면 v3와 같은 모델."""
    EXTRA = {'2층단가': ['w_2nd'], '지하마감': ['w_fin'], '외장품질': ['g_e'], '지하품질': ['g_b'],
             '난방품질': ['g_h'], '기능상태': ['g_f'], '차고칸수': ['e_car'], '품질가속': ['g_q2'],
             '건물유형': ['t_' + c for c in BLDG], '판매연도': ['y_' + c for c in YRS],
             '기초공법': ['f_' + c for c in FOUND]}

    def __init__(self, nbs, opts=(), lam=0.3):
        self.idx = {n: i for i, n in enumerate(nbs)}; self.K = len(nbs)
        self.opts, self.lam = set(opts), lam
        names = [f'land{i}' for i in range(self.K)] + [f'bld{i}' for i in range(self.K)]
        names += ['b_lot', 'w_bs', 'w_gar', 'g_q', 'g_c', 'd_age', 'd_rem']
        names += ['s_' + c for c in SALE] + ['z_' + c for c in ZONE]
        names += ['e_bath', 'e_fp', 'e_porch', 'g_k', 'g_air']
        for o in sorted(self.opts):
            names += self.EXTRA[o]
        self.pos = {n: i for i, n in enumerate(names)}

    def _init(self):
        K = self.K; th = np.zeros(len(self.pos))
        th[:K], th[K:2*K] = np.log(30), np.log(80)
        init = dict(b_lot=0.5, w_bs=0.4, w_gar=0.3, g_q=0.15, g_c=0.03, d_age=0.005, d_rem=0.002,
                    e_bath=3, e_fp=3, e_porch=1, g_k=0.05, g_air=0.05, w_2nd=1.0, w_fin=0.2, e_car=3)
        for k, v in init.items():
            if k in self.pos: th[self.pos[k]] = v
        return th

    def _predict(self, th, d):
        p = lambda n: th[self.pos[n]]
        on = lambda o: o in self.opts
        K = self.K; i = d['Neighborhood'].map(self.idx).values
        land_val = np.exp(th[:K][i]) * (d['LotArea'].values / 10000) ** p('b_lot')
        # 면적: 지상 + 지하(+마감 추가 단가) + 차고
        if on('2층단가'):
            living = d['1stFlrSF'].values + p('w_2nd') * d['2ndFlrSF'].values + d['LowQualFinSF'].values
        else:
            living = d['GrLivArea'].values
        bsmt = p('w_bs') * d['TotalBsmtSF'].values
        if on('지하마감'): bsmt = bsmt + p('w_fin') * d['bs_fin'].values
        area = living + bsmt + p('w_gar') * d['GarageArea'].values
        # 건물 품질 (로그 배수)
        q = d['OverallQual'].values - 5
        qlog = p('g_q') * q + p('g_c') * (d['OverallCond'].values - 5)
        if on('품질가속'): qlog = qlog + p('g_q2') * q ** 2
        if on('외장품질'): qlog = qlog + p('g_e') * (d['EQ'].values - 3)
        if on('지하품질'): qlog = qlog + p('g_b') * (d['BQ'].values - 3) * (d['BQ'].values > 0)
        if on('난방품질'): qlog = qlog + p('g_h') * (d['HQ'].values - 5)
        dep = np.exp(-p('d_age') * d['age'].values - p('d_rem') * d['remod_age'].values)
        total = land_val + np.exp(th[K:2*K][i]) * area * np.exp(qlog) * dep / 1000
        # 전체 배수 + 부대시설(천 달러)
        mult = p('g_k') * (d['KQ'].values - 3) + p('g_air') * (d['air'].values - 1)
        if on('기능상태'): mult = mult + p('g_f') * (d['FUNC'].values - 7)
        total = (total * np.exp(mult) + p('e_bath') * d['bath'].values
                 + p('e_fp') * d['Fireplaces'].values + p('e_porch') * d['porch'].values / 100)
        if on('차고칸수'): total = total + p('e_car') * d['GarageCars'].values
        # % 보정
        adj = (sum(p('s_' + c) * (d['SaleCondition'].values == c) for c in SALE)
               + sum(p('z_' + c) * (d['MSZoning'].values == c) for c in ZONE))
        if on('건물유형'): adj = adj + sum(p('t_' + c) * (d['BldgType'].values == c) for c in BLDG)
        if on('판매연도'): adj = adj + sum(p('y_' + c) * (d['YrSold_s'].values == c) for c in YRS)
        if on('기초공법'): adj = adj + sum(p('f_' + c) * (d['Foundation'].values == c) for c in FOUND)
        return np.log1p(np.clip(total, 1, None) * np.exp(adj) * 1000)

    def fit(self, d, yv):
        K = self.K
        def residual(th):
            pen = 0.1 * np.sqrt(self.lam) * np.r_[th[:K] - th[:K].mean(), th[K:2*K] - th[K:2*K].mean()]
            return np.r_[self._predict(th, d) - yv, pen]
        self.th = least_squares(residual, self._init(), max_nfev=5000).x
        return self

    def predict(self, d):
        return self._predict(self.th, d)

    def params(self):
        return {n: self.th[i] for n, i in self.pos.items() if not n.startswith(('land', 'bld'))}

S_train, S_test = prep_struct(train), prep_struct(test_raw)
NBS = sorted(S_train['Neighborhood'].unique())
V4 = ('지하마감',)   # 실험(7-0)으로 고른 최종 항목

In [6]:
#5 베이스 모델
from xgboost import XGBRegressor

abn = (train['SaleCondition'] != 'Normal').values
w_abn = np.where(abn, 0.5, 1.0)   # v1 5-16: 비정상 거래 가중치 0.5 (Lasso만)

def make_xgb(seed=42):
    return XGBRegressor(learning_rate=0.02, n_estimators=2500, max_depth=3, subsample=0.7,
                        colsample_bytree=0.2, min_child_weight=3, random_state=seed, n_jobs=-1)

base_models = {
    'Lasso_w': (lasso, {'lasso__sample_weight': w_abn}),
    'XGBoost': (make_xgb(), None),
    'SVR':     (svr, None),
}

In [7]:
#6 OOF 예측 (같은 분할 kf) — 다른 모델 3개 + 구조 v4
oof = pd.DataFrame(index=range(len(y)))
for name, (m, p) in base_models.items():
    t = time.time()
    oof[name] = cross_val_predict(m, X, y, cv=kf, params=p)
    print(f'{name:8s} {rmse(oof[name]):.4f}  ({time.time() - t:.0f}초)')

def struct_oof(opts, cv=kf):
    o = np.zeros(len(y))
    for tr_i, va_i in cv.split(S_train):
        o[va_i] = StructModel(NBS, opts).fit(S_train.iloc[tr_i], y.values[tr_i]).predict(S_train.iloc[va_i])
    return o

oof['구조v4'] = struct_oof(V4)
print(f'구조v4   {rmse(oof["구조v4"]):.4f}')
oof.sub(y.values, axis=0).corr().round(2)   # 오차 상관

Lasso_w  0.1092  (1초)
XGBoost  0.1119  (17초)
SVR      0.1094  (2초)
구조v4   0.1103


,Lasso_w,XGBoost,SVR,구조v4
Lasso_w,1.00,0.88,0.94,0.87
XGBoost,0.88,1.00,0.87,0.82
SVR,0.94,0.87,1.00,0.82
구조v4,0.87,0.82,0.82,1.00


In [8]:
#6-1 하이브리드: 구조 모델이 먼저 예측 → Lasso가 잔차만 보정
HYB_ALPHA = 0.0005

def fit_hybrid(S_tr, X_tr, y_tr, w_tr):
    sm = StructModel(NBS, V4).fit(S_tr, y_tr)
    resid = y_tr - sm.predict(S_tr)
    lm = make_pipeline(RobustScaler(), Lasso(alpha=HYB_ALPHA, max_iter=50000))
    lm.fit(X_tr, resid, lasso__sample_weight=w_tr)
    return sm, lm

def hybrid_oof(cv=kf):
    o = np.zeros(len(y))
    for tr_i, va_i in cv.split(X):
        sm, lm = fit_hybrid(S_train.iloc[tr_i], X.iloc[tr_i], y.values[tr_i], w_abn[tr_i])
        o[va_i] = sm.predict(S_train.iloc[va_i]) + lm.predict(X.iloc[va_i])
    return o

oof['하이브리드'] = hybrid_oof()
print(f'하이브리드 {rmse(oof["하이브리드"]):.4f}  (Lasso_w {rmse(oof["Lasso_w"]):.4f}, 구조v4 {rmse(oof["구조v4"]):.4f})')

# 바깥 분할을 바꿔도 Lasso보다 좋은지
for seed in [1, 2, 3]:
    cv = KFold(5, shuffle=True, random_state=seed)
    l = cross_val_predict(lasso, X, y, cv=cv, params={'lasso__sample_weight': w_abn})
    print(f'분할 seed {seed}: Lasso_w {rmse(l):.4f} → 하이브리드 {rmse(hybrid_oof(cv)):.4f}')

oof.sub(y.values, axis=0).corr().round(2)['하이브리드']   # 다른 모델과 오차 상관

하이브리드 0.1054  (Lasso_w 0.1092, 구조v4 0.1103)
분할 seed 1: Lasso_w 0.1082 → 하이브리드 0.1059
분할 seed 2: Lasso_w 0.1093 → 하이브리드 0.1071
분할 seed 3: Lasso_w 0.1080 → 하이브리드 0.1067


Lasso_w    0.92
XGBoost    0.84
SVR        0.87
구조v4       0.95
하이브리드      1.00
Name: 하이브리드, dtype: float64

## 7-0. 구조 모델 항목 실험
항목을 하나씩 켜서 **단독 점수**와 **스택에 넣었을 때 점수**를 같이 봐요. 단독으로 좋아져도 다른 모델이 이미 아는 정보면 스택은 안 좋아지기 때문에 둘 다 봐야 해요. 셀마다 2~4분 걸려요.

In [9]:
%%script echo skipped
#7-0a 항목 하나씩 켜기 (v3 기준)
from sklearn.linear_model import LinearRegression
meta = LinearRegression(positive=True)
others = ['Lasso_w', 'XGBoost', 'SVR']

def stack_score(struct_pred, n_seeds=10):
    D = oof[others].copy(); D['s'] = struct_pred
    return np.mean([rmse(cross_val_predict(meta, D, y, cv=KFold(5, shuffle=True, random_state=s)))
                    for s in range(n_seeds)])

v3_pred = struct_oof(())
b_solo, b_stack = rmse(v3_pred), stack_score(v3_pred)
rows = [{'항목': 'v3 (기준)', '단독': round(b_solo, 4), '스택': round(b_stack, 4)}]
for opt in ['2층단가', '지하마감', '외장품질', '지하품질', '난방품질', '기능상태',
            '건물유형', '판매연도', '기초공법', '차고칸수', '품질가속']:
    pr = struct_oof((opt,))
    s1, s2 = rmse(pr), stack_score(pr)
    rows.append({'항목': opt, '단독': round(s1, 4), '단독 변화': round(s1 - b_solo, 4),
                 '스택': round(s2, 4), '스택 변화': round(s2 - b_stack, 4)})
pd.DataFrame(rows)

skipped


In [10]:
%%script echo skipped
#7-0b 좋았던 항목 조합 + 분할을 바꿔도 유지되는지
combos = {'v3': (), '지하마감': ('지하마감',), '+기능상태': ('지하마감', '기능상태'),
          '+품질가속 (최종)': V4, '+지하품질': V4 + ('지하품질',)}
for name, c in combos.items():
    pr = struct_oof(c)
    print(f'{name:14s} 단독 {rmse(pr):.4f} | 스택 {stack_score(pr):.4f}')

print()
for seed in [1, 2, 3]:
    k = KFold(5, shuffle=True, random_state=seed)
    print(f'분할 seed {seed}: v3 {rmse(struct_oof((), k)):.4f} → v4 {rmse(struct_oof(V4, k)):.4f}')

skipped


In [11]:
#7-0c v4 해석: 각 항목이 집값에 주는 영향 (발표용)
th = StructModel(NBS, V4).fit(S_train, y.values).params()
print(f"지하 1sqft = 지상의 {th['w_bs']:.2f}배", end='')
print(f", 마감된 지하는 여기에 +{th['w_fin']:.2f}배" if 'w_fin' in th else '')
print(f"차고 1sqft = 지상의 {th['w_gar']:.2f}배")

g_q2 = th.get('g_q2', 0.0)   # 품질가속이 꺼져 있으면 0
for qv in [3, 5, 7, 9, 10]:
    d = qv - 5
    print(f"OverallQual {qv:2d}: 건물값 x{np.exp(th['g_q'] * d + g_q2 * d * d):.2f} (5등급 대비)")

if 'g_f' in th:
    print(f"Functional 감점 1단계당 전체 가격 {-np.expm1(-th['g_f']) * 100:.1f}% 하락")

지하 1sqft = 지상의 0.41배, 마감된 지하는 여기에 +0.37배
차고 1sqft = 지상의 0.56배
OverallQual  3: 건물값 x0.82 (5등급 대비)
OverallQual  5: 건물값 x1.00 (5등급 대비)
OverallQual  7: 건물값 x1.22 (5등급 대비)
OverallQual  9: 건물값 x1.49 (5등급 대비)
OverallQual 10: 건물값 x1.64 (5등급 대비)


In [12]:
#7 스태킹 vs 단순 평균 (스태킹 분할 10가지)
from sklearn.linear_model import LinearRegression

cols = ['XGBoost', 'SVR', '하이브리드']
meta = LinearRegression(positive=True)

stack_s = [rmse(cross_val_predict(meta, oof[cols], y, cv=KFold(5, shuffle=True, random_state=s)))
           for s in range(10)]
print(f'스태킹    {np.mean(stack_s):.4f} (±{np.std(stack_s):.4f}, 10가지 분할)')
print(f'단순 평균 {rmse(oof[cols].mean(axis=1)):.4f}')

meta.fit(oof[cols], y)
print('스태킹 가중치:', dict(zip(cols, meta.coef_.round(2))),
      '| 합', meta.coef_.sum().round(2), '| 절편', round(meta.intercept_, 3))

스태킹    0.1034 (±0.0002, 10가지 분할)
단순 평균 0.1036
스태킹 가중치: {'XGBoost': np.float64(0.21), 'SVR': np.float64(0.27), '하이브리드': np.float64(0.54)} | 합 1.01 | 절편 -0.132


## 8. 최종 학습 & 제출
- XGBoost는 seed 3개 평균으로 test 예측 흔들림을 줄여요.
- Id 2550 규칙(초대형 + Partial → 524·1299 평균 가격)은 v1과 동일하게 유지해요.
- `submission_v4_stack.csv`와 `submission_v4_blend.csv`를 만들어요. 지금까지 stack과 blend는 리더보드에서 0.0001~0.0005 차이로 엎치락뒤치락했으니, **두 파일 모두 제출**해서 비교하세요.

In [13]:
#8 최종 학습 + 제출 파일 2개
test_preds = pd.DataFrame({
    'Lasso_w': lasso.fit(X, y, lasso__sample_weight=w_abn).predict(X_test),
    'XGBoost': np.mean([make_xgb(s).fit(X, y).predict(X_test) for s in [42, 7, 2026]], axis=0),
    'SVR':     svr.fit(X, y).predict(X_test),
    '구조v4':  StructModel(NBS, V4).fit(S_train, y.values).predict(S_test),
})
sm_full, lm_full = fit_hybrid(S_train, X, y.values, w_abn)
test_preds['하이브리드'] = sm_full.predict(S_test) + lm_full.predict(X_test)

# Id 2550 규칙
twin_log = np.log1p(train_raw.loc[train_raw['Id'].isin([524, 1299]), 'SalePrice']).mean()
special = ((test_raw['GrLivArea'] > 4000) & (test_raw['SaleCondition'] == 'Partial')).values

outputs = {'submission_v4_stack.csv': meta.predict(test_preds[cols]),
           'submission_v4_blend.csv': test_preds[cols].mean(axis=1).values}
for fname, pred_log in outputs.items():
    pred_log = pred_log.copy()
    pred_log[special] = twin_log
    sub = pd.DataFrame({'Id': test_raw['Id'], 'SalePrice': np.expm1(pred_log)})
    sub.to_csv(fname, index=False)
    print(fname, sub.shape, '| 2550:', round(sub.loc[sub['Id'] == 2550, 'SalePrice'].iloc[0]))

# 모델 간 test 예측 상관 (너무 낮은 모델이 있으면 이상 신호)
test_preds.corr().round(3)

submission_v4_stack.csv (1459, 2) | 2550: 171930
submission_v4_blend.csv (1459, 2) | 2550: 171930


,Lasso_w,XGBoost,SVR,구조v4,하이브리드
Lasso_w,1.000,0.991,0.995,0.989,0.993
XGBoost,0.991,1.000,0.989,0.987,0.989
SVR,0.995,0.989,1.000,0.985,0.989
구조v4,0.989,0.987,0.985,1.000,0.997
하이브리드,0.993,0.989,0.989,0.997,1.000


In [14]:
#9 점검: test 예측 분포가 train과 비슷한지
print('train  log 가격:', y.describe()[['mean', 'std', 'min', 'max']].round(3).to_dict())
for f in outputs:
    p = np.log1p(pd.read_csv(f)['SalePrice'])
    print(f'{f:22s}', p.describe()[['mean', 'std', 'min', 'max']].round(3).to_dict())

train  log 가격: {'mean': 12.024, 'std': 0.4, 'min': 10.46, 'max': 13.534}
submission_v4_stack.csv {'mean': 12.011, 'std': 0.401, 'min': 10.563, 'max': 13.279}
submission_v4_blend.csv {'mean': 12.012, 'std': 0.395, 'min': 10.628, 'max': 13.273}


In [15]:
#10 가설 검증: 상업지역 코너 땅 6채만 +10%
sub = pd.read_csv('submission_v4_blend.csv')
corner = ((test_raw['Neighborhood'] == 'IDOTRR') & (test_raw['MSZoning'] == 'C (all)')
          & (test_raw['LotConfig'] == 'Corner')).values
print('대상:', corner.sum(), '채', test_raw.loc[corner, 'Id'].tolist())
sub.loc[corner, 'SalePrice'] *= 1.10
sub.to_csv('submission_v4_blend_corner10.csv', index=False)

대상: 6 채 [1603, 1823, 2218, 2219, 2581, 2792]
